# 071 — Sensores, series y percepción en el borde

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Ventanas deslizantes:** el flujo del sensor se corta en ventanas de `w` muestras con
salto `h`: `n = 1 + ⌊(T − w)/h⌋`. `w` debe cubrir el fenómeno (un paso ~1 s, una caída
~2 s); `h` fija la latencia de decisión. Con solapamiento hay más ejemplos — y riesgo de
fuga si se parte train/test por ventana en vez de **por sujeto o sesión**.

**Features temporales:** media, desviación, **RMS** (`√(Σx²/n)`), cruces por cero (ZCR),
energía por banda de la FFT. Un clasificador clásico sobre estas features es un baseline
duro en HAR con cómputo mínimo.

**Cuantización int8:** `x ≈ s·(q − z)`; simétrica: `z = 0`, `s = max|x|/127`. 4× menos
memoria y aritmética entera (clave en MCU sin FPU); caída típica ~1 punto
(post-entrenamiento), menos con QAT.

**TinyML:** inferencia en microcontroladores (64-256 kB RAM, mW) con runtimes como
TFLite Micro / LiteRT; *pruning*, *distillation* y *duty-cycling* (un detector diminuto
despierta al modelo grande). Motivos del borde: latencia, energía, autonomía y
**privacidad** — la señal cruda nunca sale del dispositivo.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Conteo de ventanas.** Una señal de 60 s muestreada a 25 Hz se corta con
ventanas de 4 s y hop de 2 s. (a) Calcula T, w, h en muestras y el número de ventanas.
(b) ¿Cuántas ventanas quedan sin solapamiento (h = w)? ¿Qué se gana y se arriesga al
solapar?

**Ejercicio 2 — Features a mano.** Calcula media, RMS y cruces por cero de
`A = [1, −1, 1, −1, 1, −1, 1, −1]` y de `B = [1, 1, 1, 1, −1, −1, −1, −1]`. ¿Qué feature
distingue a las dos señales y qué información captura?

**Ejercicio 3 — Cuantización int8.** Los pesos de un modelo viven en [−1.6, 1.6].
(a) Calcula la escala simétrica `s = max|x|/127`. (b) Cuantiza `x = 0.5`, dequantízalo y
calcula el error. (c) Un modelo de 120 000 parámetros: ¿cuánta memoria ocupan los pesos en
float32 y en int8? ¿Cabe en 256 kB de flash?

**Ejercicio 4 — En código.** Implementa `n_ventanas`, las tres features y
`quantize/dequantize`, y verifica los ejercicios 1-3.


In [ ]:
# TODO: ejecuta run_lab("robotics", seed=71)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ
# Ejercicio 4: ventanas, features y cuantización
def n_ventanas(T, w, h):
    return None

def features(x):
    # devuelve (media, rms, cruces_por_cero)
    return None

def quantize(x, s):
    # int8 simétrico: round(x/s), recortado a [-127, 127]
    return None

# verifica: n_ventanas(1500, 100, 50), features de A y B, quantize(0.5, 1.6/127)


## Reflexión

1. Tu HAR da 97 % validando por ventanas al azar y 78 % validando por sujeto. ¿Cuál de los
   dos números reportas, y qué mecanismo exacto produce la brecha?
2. Un detector de caídas debe decidir en menos de 1 s, pero la batería debe durar meses.
   ¿Qué combinación de hop, duty-cycle y confirmación multi-ventana propones y qué
   trade-off acepta cada elección?
3. Para un sensor de audio doméstico, ¿qué ventaja concreta de privacidad da inferir en el
   borde, y qué telemetría podrías subir a la nube sin traicionarla?
